[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jsonmen/bias-and-variance/blob/main/nlp/SentimentAnalysis/IMDB50K/BERTSentimentAnalysis.ipynb)

# Abstract

- Goal: Fine-Tune BERT for Sentiment Analysis task and reach good accuracy score

- Dataset: [IMDB Dataset of 50K Movie Reviews (Kaggle)](https://www.kaggle.com/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews)

- Project Details:

    I'm using bert-base-uncased from google-bert with transfromers library. This model fine-tuned to analyze sentiment on IMDB Movie Reviews dataset. Within train all model gradients be frozen except pooler layers.

- Best result: 0.8642 (Accuracy Score of Fine-Tuned BERT)

- Sections:
    - [Imports](#Imports)
    - [Setup Model](#Setup-Model)
    - [Dataset](#Dataset)
    - [Model Train](#Model-Train)
    - [Prediction](#Prediction)  
        - [Load Model](#Load-Model)  
        - [Setup Form](#Setup-Form)  
        - [Prediction Form](#Prediction-Form)

## Problems & Solutions
There was no problems. It's easy just load pre-trained model, prepare dataset, train model and you get a good result. Only one think that i need to write, model maybe can have greater accuracy score if i unlock all layers but i think its be only +2-5% to accuracy score but train time be greater and more computationally expensive, 5% isn't big deal for sentiment analysis task

# Download & Install Dependencies

In [ ]:
# Install modules (if needed)
!pip install ipywidgets ipython evaluate huggingface-hub transformers datasets pandas # torch

In [2]:
# Dataset Downloading
!mkdir data
!curl -L -o ./data/dataset.zip https://www.kaggle.com/api/v1/datasets/download/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews
!unzip ./data/dataset.zip -d ./data
!rm ./data/dataset.zip
!mv ./data/IMDB\ Dataset.csv ./data/imdb_dataset.csv

mkdir: cannot create directory ‘data’: File exists
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100 25.7M  100 25.7M    0     0  7371k      0  0:00:03  0:00:03 --:--:-- 9592k
Archive:  ./data/dataset.zip
  inflating: ./data/IMDB Dataset.csv  


In [3]:
# Model Downloading
!mkdir models
from huggingface_hub import snapshot_download

PROJECT_NAME = "SentimentAnalysisIMDB50K"
MODEL_FOLDER = "bert"
repo_id = f"jsonmen/{PROJECT_NAME}"

snapshot_download(
    repo_id=repo_id,
    local_dir="./models",
    allow_patterns=[f"{MODEL_FOLDER}/*"],
    token=False  # No token needed for public repos
)
print(f"Downloaded models folder from {repo_id} to ./models")

mkdir: cannot create directory ‘models’: File exists


Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

Downloaded models folder from jsonmen/SentimentAnalysisIMDB50K to ./models


In [4]:
# For colab users (local library files download)
!curl -L -o ./setup_text_preprocessing.py https://raw.githubusercontent.com/jsonmen/bias-and-variance/refs/heads/main/SentimentAnalysis/setup_text_preprocessing.py
!curl -L -o ./text_preprocessing.py https://raw.githubusercontent.com/jsonmen/bias-and-variance/refs/heads/main/SentimentAnalysis/text_preprocessing.py

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  2939  100  2939    0     0  14300      0 --:--:-- --:--:-- --:--:-- 14266
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  3234  100  3234    0     0  15335      0 --:--:-- --:--:-- --:--:-- 15400


# Imports

In [5]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer, Trainer, TrainingArguments, DataCollatorWithPadding
from datasets import Dataset
from evaluate import load as load_metric
import pandas as pd
import ipywidgets as widgets
from IPython.display import clear_output, display
import torch
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ["WANDB_DISABLED"] = "true"

2025-07-22 02:42:35.300270: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1753144955.340890   38047 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1753144955.353750   38047 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1753144955.427313   38047 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1753144955.427323   38047 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1753144955.427325   38047 computation_placer.cc:177] computation placer alr

# Setup Model 

A few words about BERT:

BERT stands for Bidirectional Encoder Representations from Transformers.
It consists of multiple Transformer Encoder layers stacked together.

For my task — sentiment analysis (which is just classification from the model's perspective) — additional classification layers are added on top of BERT's outputs.

Here’s an image that helps to make sense of how BERT works:

<img src="https://towardsdatascience.com/wp-content/uploads/2024/05/1Qww2aaIdqrWVeNmo3AS0ZQ.png" alt="Comparison of BERT with other models" width="500"/>

And here’s BERT’s architecture during the pretraining phase (Masked Language Modeling — the original task it was trained on):

<img src="https://miro.medium.com/v2/resize:fit:876/0*ViwaI3Vvbnd-CJSQ.png" alt="BERT architecture for masked language modeling" width="500"/>

In [6]:
model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased", 
    num_labels=2,
)
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [7]:
model

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e

In [8]:
for name, param in model.bert.named_parameters():
    param.requires_grad = False

for name, param in model.bert.named_parameters():
    if "pooler" in name:
        param.requires_grad = True

In [9]:
model

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e

# Dataset

In [10]:
sentiment2label = {"positive": 1, "negative": 0}
label2sentiment = {1: "positive", 0: "negative"}

In [11]:
df = pd.read_csv("./data/imdb_dataset.csv")
df["label"] = df["sentiment"].map(sentiment2label)
dataset = Dataset.from_pandas(df.drop("sentiment", axis=1))

In [12]:
def tokenize_dataset(dataset):
    return tokenizer(
        dataset["review"],
        max_length=512,
        truncation=True,
        padding='max_length',
        return_tensors='pt'
    )
    
# Tokenize dataset
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
tokenized_dataset = dataset.map(tokenize_dataset, batched=True)

# Split into train and validation sets
train_test_split = tokenized_dataset.train_test_split(test_size=0.2)
train_dataset = train_test_split["train"]
eval_dataset = train_test_split["test"]

# Set format for PyTorch
tokenized_dataset.set_format("torch", columns=["input_ids", "attention_mask", "token_type_ids", "label"])

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

# Model Train

In [18]:
training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=3e-4,
    per_device_train_batch_size=16,
    num_train_epochs=5,
    eval_strategy="epoch",            # Evaluate at the end of each epoch
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    report_to="none"
)

In [19]:
def compute_metrics(eval_pred):
    metric = load_metric("accuracy")
    logits, labels = eval_pred
    predictions = logits.argmax(axis=-1)
    return metric.compute(predictions=predictions, references=labels)

In [20]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,  # For evaluation metrics
)

In [21]:
# Train the model
trainer.train()

# Save the model and tokenizer
model.save_pretrained("./models/bert")
tokenizer.save_pretrained("./models/bert")

Epoch,Training Loss,Validation Loss,Accuracy
1,0.378900,0.335272,0.853600
2,0.365500,0.356906,0.848400
3,0.348800,0.331851,0.859800
4,0.345200,0.320628,0.862700
5,0.357400,0.317949,0.864200


('./imdb_bert_model/tokenizer_config.json',
 './imdb_bert_model/special_tokens_map.json',
 './imdb_bert_model/vocab.txt',
 './imdb_bert_model/added_tokens.json',
 './imdb_bert_model/tokenizer.json')

# Prediction

## Load Model

In [13]:
# load trained model and tokenizer from file

trained_tokenizer = AutoTokenizer.from_pretrained("./models/bert")
trained_model = AutoModelForSequenceClassification.from_pretrained("./models/bert")

## Setup Form

In [14]:
text_field = widgets.Text(
    value='',
    placeholder='Type something',
    description='Text:',
    disabled=False   
)
submit_button = widgets.Button(description="Analyze Sentiment")
output = widgets.Output()
def predict_sentiment(text):
    inputs = trained_tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=512,
        padding=True
    )

    with torch.no_grad():
        outputs = trained_model(**inputs)
        logits = outputs.logits
        predicted_class_id = logits.argmax().item()

    sentiment_word = label2sentiment[predicted_class_id]
    return sentiment_word

def form_fn(b):
    text = text_field.value
    sentiment_word = predict_sentiment(text)
    with output:
        clear_output()
        print(f"Your text has a {sentiment_word} sentiment")

submit_button.on_click(form_fn)
form = widgets.VBox([text_field, submit_button, output])

## Prediction Form

In [15]:
display(form)